In [ ]:
CREATE OR ALTER VIEW dbo.vw__int__stg_STIP__filtered_v1
AS
SELECT
    CASE
        WHEN p.CurrManufacturerCd IN ('MMC','APL','PVL','NAVM','CRI','COLABEL') THEN 'Medline Brand'
        ELSE 'Non-Medline'
    END AS Manufacturer,
    p.FiscalYearPeriodDate,
    p.SalesTypeCd,
    p.CurrProdDivCd,
    p.CurrSoldToSalesOfficeCd,
    p.CurrMatGrpCd,

    COALESCE(p.GrossSalesAmount, 0) AS GrossSalesAmount_NZ,
    COALESCE(p.CostOfGoodsSoldAmount, 0) AS CostOfGoodsSoldAmount_NZ,
    COALESCE(p.GroupRebateAccrualAmount, 0) AS GroupRebateAccrualAmount_NZ,
    COALESCE(p.CorporateRebateAccrualAmount, 0) AS CorporateRebateAccrualAmount_NZ,
    COALESCE(p.CustomerIncentiveRebateAmt, 0) AS CustomerIncentiveRebateAmt_NZ
FROM dbo.SoldToInvcProfitabilityModel p
WHERE p.FiscalYearPeriodDate IN ('2025010','2025011','2025012')
  AND p.CurrProdDivCd NOT IN ('01','11','12','13','83','90','91','92','93','94','95','96','99')
  AND p.CurrSoldToSalesOfficeCd = 'GL'
  AND (
        (p.CurrDistributorPrefixCode <> 'CM' AND p.SalesTypeCd = 'DS')
        OR p.SalesTypeCd = 'DRC'
      );


In [ ]:
CREATE OR ALTER VIEW dbo.vw__int__stg_STIP__mat_joined_v1
AS
SELECT
    f.Manufacturer,
    f.FiscalYearPeriodDate,
    f.SalesTypeCd,
    f.CurrProdDivCd,
    f.CurrSoldToSalesOfficeCd,
    f.CurrMatGrpCd,

    f.GrossSalesAmount_NZ,
    f.CostOfGoodsSoldAmount_NZ,
    f.GroupRebateAccrualAmount_NZ,
    f.CorporateRebateAccrualAmount_NZ,
    f.CustomerIncentiveRebateAmt_NZ,

    m.MaterialGroupDescription,
    m.ProductDivisionCode,
    m.ProductDivisionDescription
FROM dbo.vw__int__stg_STIP__filtered_v1 f
INNER JOIN dbo.MaterialMaster m
    ON m.MaterialGroupCode = f.CurrMatGrpCd
   AND m.ProductDivisionCode IS NOT NULL
   AND LTRIM(RTRIM(m.ProductDivisionCode)) <> '';


In [ ]:
CREATE OR ALTER VIEW dbo.vw__int__agg_STIP__core_v1
AS
SELECT
    j.Manufacturer,
    j.FiscalYearPeriodDate,
    j.SalesTypeCd,
    j.CurrProdDivCd,
    j.CurrSoldToSalesOfficeCd,
    j.CurrMatGrpCd,

    SUM(j.GrossSalesAmount_NZ) AS GrossSales,
    SUM(j.CostOfGoodsSoldAmount_NZ) AS CostOfGoodsSold,
    -SUM(j.GroupRebateAccrualAmount_NZ) AS GroupRebateAccrualAmount,
    -SUM(j.CorporateRebateAccrualAmount_NZ) AS CorporateRebateAccrualAmount,
    -SUM(j.CustomerIncentiveRebateAmt_NZ) AS CustomerIncentiveRebateAmt,

    j.MaterialGroupDescription,
    j.ProductDivisionCode,
    j.ProductDivisionDescription
FROM dbo.vw__int__stg_STIP__mat_joined_v1 j
GROUP BY
    j.Manufacturer,
    j.FiscalYearPeriodDate,
    j.SalesTypeCd,
    j.CurrProdDivCd,
    j.CurrSoldToSalesOfficeCd,
    j.CurrMatGrpCd,
    j.MaterialGroupDescription,
    j.ProductDivisionCode,
    j.ProductDivisionDescription;


In [ ]:
CREATE OR ALTER VIEW dbo.vw__CM__STIP_summary_v1
AS
SELECT *
FROM dbo.vw__int__agg_STIP__core_v1;
